In [1]:
import json
import os
from sentence_transformers import SentenceTransformer
import chromadb

In [2]:
# Paths
CHUNKS_PATH   = "data/chunks/all_chunks.json"
VECTORSTORE   = "data/vectorstore"          # ChromaDB writes to disk here
COLLECTION    = "pet_care"                  # name of the collection inside ChromaDB


In [3]:
# Load all chunks

with open(CHUNKS_PATH, encoding="utf-8") as f:
    all_chunks = json.load(f)

print(f"Loaded {len(all_chunks)} chunks")

Loaded 1462 chunks


In [4]:
# Initialize embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding model ready")

#initialize ChromaDB client and collection
client = chromadb.PersistentClient(path=VECTORSTORE)

collection = client.get_or_create_collection(
    name=COLLECTION,
    metadata={"hnsw:space": "cosine"}   # cosine similarity — standard for text
)

print(f"ChromaDB collection '{COLLECTION}' ready")
print(f"Documents already in collection: {collection.count()}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model ready
ChromaDB collection 'pet_care' ready
Documents already in collection: 0


In [5]:
# Embed and store — skip if already exists
# Check existing IDs once up front — cheaper than querying per chunk

existing_ids = set(collection.get(include=[])["ids"])

to_embed      = [c for c in all_chunks if c["chunk_id"] not in existing_ids]
already_there = len(all_chunks) - len(to_embed)

print(f"Skipping {already_there} already-embedded chunks")
print(f"Embedding {len(to_embed)} new chunks...")

# Embed in batches — prevents memory issues on large runs
# 64 is a safe batch size for all-MiniLM-L6-v2 on a CPU
BATCH_SIZE = 64

for i in range(0, len(to_embed), BATCH_SIZE):
    batch = to_embed[i : i + BATCH_SIZE]

    # Extract the fields ChromaDB needs
    ids       = [c["chunk_id"] for c in batch]
    texts     = [c["text"]     for c in batch]
    metadatas = [
        {
            "title":   c["title"],
            "url":     c["url"],
            "source":  c["source"],
            "species": c["species"],
            "topic":   c["topic"]
        }
        for c in batch
    ]

    # Embed all texts in this batch at once — faster than one-by-one
    embeddings = model.encode(texts, show_progress_bar=False).tolist()

    # Store in ChromaDB: id, embedding, original text, metadata
    collection.add(
        ids        = ids,
        embeddings = embeddings,
        documents  = texts,
        metadatas  = metadatas
    )

    print(f"  Embedded chunks {i+1}–{min(i+BATCH_SIZE, len(to_embed))} of {len(to_embed)}")

print(f"\nDone. Collection now contains {collection.count()} documents.")

Skipping 0 already-embedded chunks
Embedding 1462 new chunks...
  Embedded chunks 1–64 of 1462
  Embedded chunks 65–128 of 1462
  Embedded chunks 129–192 of 1462
  Embedded chunks 193–256 of 1462
  Embedded chunks 257–320 of 1462
  Embedded chunks 321–384 of 1462
  Embedded chunks 385–448 of 1462
  Embedded chunks 449–512 of 1462
  Embedded chunks 513–576 of 1462
  Embedded chunks 577–640 of 1462
  Embedded chunks 641–704 of 1462
  Embedded chunks 705–768 of 1462
  Embedded chunks 769–832 of 1462
  Embedded chunks 833–896 of 1462
  Embedded chunks 897–960 of 1462
  Embedded chunks 961–1024 of 1462
  Embedded chunks 1025–1088 of 1462
  Embedded chunks 1089–1152 of 1462
  Embedded chunks 1153–1216 of 1462
  Embedded chunks 1217–1280 of 1462
  Embedded chunks 1281–1344 of 1462
  Embedded chunks 1345–1408 of 1462
  Embedded chunks 1409–1462 of 1462

Done. Collection now contains 1462 documents.
